## Feature Engineering – RFM Analysis

Bu adımda temizlenmiş Online Retail verisi kullanılarak müşteri bazlı
Recency, Frequency ve Monetary metrikleri hesaplanmış ve müşteriler
davranışlarına göre segmentlere ayrılmıştır.




## Google Drive Bağlantısı

Bu adımda, proje dosyalarına erişmek ve çıktıların Drive’a kaydedilebilmesi için Google Drive bağlanmaktadır.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Proje Klasör Yapısının Oluşturulması

Bu adımda, işlenmiş veri ve notebook dosyaları için gerekli klasör yapısı oluşturulmuştur.


In [2]:
import os

base_path = '/content/drive/MyDrive/online-retail-analysis'
os.makedirs(f'{base_path}/data/processed', exist_ok=True)
os.makedirs(f'{base_path}/notebooks', exist_ok=True)


## Temizlenmiş Verinin Yüklenmesi

Bir önceki adımda (01_data_cleaning) oluşturulan temizlenmiş veri seti analiz için tekrar yüklenmiştir.


In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/online-retail-analysis/data/processed/cleaned_online_retail.csv')
df.head()



,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceMonth,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010-12,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010-12,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12,20.34


## Feature Engineering – Monetary ve Reference Date Oluşturma


In [4]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
reference_date


Timestamp('2011-12-10 12:50:00')

## RFM Metriklerinin Hesaplanması


In [5]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


## RFM Skorlarının Oluşturulması


In [6]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)


In [7]:
rfm.head()


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
CustomerID,,,,,,,
12346.0,326,1,77183.60,1,1,5,115
12347.0,2,7,4310.00,5,5,5,555
12348.0,75,4,1797.24,2,4,4,244
12349.0,19,1,1757.55,4,1,4,414
12350.0,310,1,334.40,1,1,2,112


In [8]:
import os
os.makedirs('data/processed', exist_ok=True)


In [9]:
rfm.to_csv('data/processed/rfm_table.csv')
